# NeuroScale++ — ANN-to-SNN Adaptive Inference on Google Colab

**Full pipeline (~25–35 min on T4 GPU):**
1. **Phase 1** — Train ResNet-20 on CIFAR-10 (30 epochs, fast cosine schedule)
2. **Phase 2** — Convert ANN→SNN, profile per-sample scaling laws, fit (α, β, γ)
3. **Phase 3** — Joint train: Complexity Predictor + Multi-Exit SNN branches (20 epochs)
4. **Evaluation** — Adaptive inference, energy savings, plots

> ⚡ **Before running:** `Runtime → Change runtime type → GPU (T4)`

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scipy'])

import os, time, copy, warnings
from pathlib import Path
from collections import OrderedDict
from typing import Optional
warnings.filterwarnings('ignore')

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, TensorDataset
import torchvision, torchvision.transforms as transforms
from scipy.optimize import curve_fit
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')  # non-interactive — saves to file, works everywhere
import matplotlib.pyplot as plt

# ── Device: MPS (Apple Silicon) > CUDA > CPU ──────────────────────────────────
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    torch.backends.cudnn.benchmark = True

# ── Paths: works on both Mac (local) and Colab ────────────────────────────────
BASE = Path('/content') if Path('/content').exists() else Path('.')
CKPT = BASE / 'checkpoints'; CKPT.mkdir(parents=True, exist_ok=True)
DATA = BASE / 'data';        DATA.mkdir(parents=True, exist_ok=True)
PLOT = BASE / 'plots';       PLOT.mkdir(parents=True, exist_ok=True)

torch.manual_seed(42); np.random.seed(42)
print(f'Checkpoints : {CKPT}')
print(f'Data        : {DATA}')
print(f'Plots       : {PLOT}')
print('Setup complete!')

Device: cpu


FileNotFoundError: [Errno 2] No such file or directory: '/content/checkpoints'

In [ ]:
import gc
gc.collect()
if device.type == 'cuda': torch.cuda.empty_cache()

# ── Memory monitor helper — run anytime to see current usage ──────────────────
def mem_report():
    gc.collect()
    if device.type == 'cuda':
        alloc  = torch.cuda.memory_allocated() / 1024**3
        reserv = torch.cuda.memory_reserved()  / 1024**3
        total  = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f'GPU  allocated : {alloc:.2f} GB')
        print(f'GPU  reserved  : {reserv:.2f} GB')
        print(f'GPU  free      : {total - reserv:.2f} GB')
    elif device.type == 'mps':
        print(f'MPS memory allocated: {torch.mps.current_allocated_memory()/1024**3:.2f} GB')
    import psutil, os
    proc = psutil.Process(os.getpid())
    ram_gb = proc.memory_info().rss / 1024**3
    print(f'CPU RAM (process): {ram_gb:.2f} GB')

def free_memory():
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache(); torch.cuda.synchronize()
    elif device.type == 'mps':
        torch.mps.empty_cache()
    print('Memory freed.')

def show_plot(fig, path):
    """Save plot and display inline if in Jupyter, else just save."""
    fig.savefig(path, dpi=120, bbox_inches='tight')
    try:
        from IPython.display import display, Image
        display(Image(filename=str(path)))
    except Exception:
        print(f'Plot saved: {path}')
    plt.close(fig)

print('Memory helpers ready. Call mem_report() or free_memory() anytime.')

## Configuration
Tuned for speed. Change `epochs`/`num_samples` to trade speed vs accuracy.

In [ ]:
CFG = {
    'dataset' : {'name':'cifar10','num_classes':10,'image_size':32},
    'model'   : {'ann':'resnet20','batch_size':128,'epochs':15,
                 'lr':0.1,'weight_decay':1e-4,'momentum':0.9},
    'conversion':{'calibration_samples':512,'percentile':99.9},
    'snn'     : {'timesteps':[1,2,4,8,16,32],'max_timestep':32,
                 'exit_points':[4,8,16,32]},
    'profiling': {'num_samples':2000,'batch_size':128},
    'predictor': {'hidden_dims':[64,128,64],'lr':1e-3,'epochs':10,'batch_size':128},
    'scaling_law':{'target_accuracy':0.90},
    'inference': {'confidence_threshold':0.9,'energy_weight':0.5},
}
print('Config OK | ANN epochs:', CFG['model']['epochs'],
      '| SNN max T:', CFG['snn']['max_timestep'],
      '| Profiling samples:', CFG['profiling']['num_samples'])

## Module Definitions
All source code is inlined here — no external files needed.

### Datasets

In [ ]:
def get_cifar10_loaders(data_dir, batch_size=128, num_workers=2, calib_only=False, calib_n=512):
    mean, std = [0.4914,0.4822,0.4465], [0.2023,0.1994,0.2010]
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean,std)])
    base_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean,std)])

    train_ds = torchvision.datasets.CIFAR10(str(data_dir), train=True,  download=True, transform=train_tf)
    test_ds  = torchvision.datasets.CIFAR10(str(data_dir), train=False, download=True, transform=test_tf)

    kw = dict(num_workers=num_workers, pin_memory=(device.type=='cuda'))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  drop_last=True, **kw)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, **kw)

    if calib_only:
        calib_ds = torchvision.datasets.CIFAR10(str(data_dir), train=True, download=False, transform=base_tf)
        idx = torch.randperm(len(calib_ds))[:calib_n]
        calib_loader = DataLoader(Subset(calib_ds, idx), batch_size=64, shuffle=False, **kw)
        return calib_loader
    return train_loader, test_loader

print('Dataset helpers defined.')

### Model — ResNet-20 (CIFAR)

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class ResNetCIFAR(nn.Module):
    def __init__(self, num_blocks, num_classes=10):
        super().__init__()
        self.in_planes = 16
        self.conv1  = nn.Conv2d(3, 16, 3, stride=1, padding=1, bias=False)
        self.bn1    = nn.BatchNorm2d(16)
        self.layer1 = self._make_layer(16, num_blocks[0], 1)
        self.layer2 = self._make_layer(32, num_blocks[1], 2)
        self.layer3 = self._make_layer(64, num_blocks[2], 2)
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc      = nn.Linear(64, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
    def _make_layer(self, planes, nb, stride):
        layers = []
        for s in [stride]+[1]*(nb-1):
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out); out = self.layer2(out); out = self.layer3(out)
        out = self.avgpool(out)
        return self.fc(out.view(out.size(0),-1))

def resnet20(num_classes=10): return ResNetCIFAR([3,3,3], num_classes)

print('ResNet-20 defined. Params:', sum(p.numel() for p in resnet20().parameters()))

### Spiking Neurons & Layers

In [ ]:
class IFNeuron(nn.Module):
    def __init__(self, threshold=1.0):
        super().__init__()
        self.threshold = threshold
        self.mem = None
    def reset(self):
        if self.mem is not None: self.mem.zero_()  # reuse buffer, no realloc
    def forward(self, x):
        if self.mem is None: self.mem = torch.zeros_like(x)
        self.mem = self.mem + x
        spikes = (self.mem >= self.threshold).float()
        self.mem = self.mem - spikes * self.threshold
        return spikes

class SpikingConv2d(nn.Module):
    def __init__(self, in_c, out_c, k, stride=1, padding=0, bias=False, threshold=1.0):
        super().__init__()
        self.conv   = nn.Conv2d(in_c, out_c, k, stride=stride, padding=padding, bias=bias)
        self.neuron = IFNeuron(threshold)
    def reset(self): self.neuron.reset()
    def forward(self, x): return self.neuron(self.conv(x))

class SpikingBN(nn.Module):
    def __init__(self, nf):
        super().__init__()
        self.bn = nn.BatchNorm2d(nf)
        self.bn.eval()
    def forward(self, x): return self.bn(x)

print('Spiking neurons defined.')

### ANN→SNN Conversion (Threshold Balancing)

In [ ]:
class ThresholdBalancer:
    def __init__(self, percentile=99.9):
        self.percentile = percentile
        self.thresholds = OrderedDict()
        self._hooks = []
        # Store only running max-values per layer, not full activation tensors
        self._max_vals = {}

    def compute_thresholds(self, model, calib_loader, device):
        model.to(device).eval()
        self._max_vals = {}; self._hooks = []
        # Hook stores only the per-batch 99.9th-percentile, not raw tensors
        for name, m in model.named_modules():
            if isinstance(m, nn.ReLU):
                def _hook(mod, inp, out, n=name):
                    v = out.detach().float()
                    q = torch.quantile(v.flatten(), self.percentile/100).item() \
                        if self.percentile < 100 else v.max().item()
                    # Keep running max across batches
                    self._max_vals[n] = max(self._max_vals.get(n, 0.0), q)
                self._hooks.append(m.register_forward_hook(_hook))
        with torch.no_grad():
            for imgs, _ in calib_loader:
                model(imgs.to(device))
        for h in self._hooks: h.remove()
        self.thresholds = OrderedDict()
        for name, val in self._max_vals.items():
            self.thresholds[name] = max(val, 1e-5)
        self._max_vals = {}  # free immediately
        return self.thresholds

def convert_ann_to_snn_weights(ann_model, thresholds):
    """Normalize ANN weights for SNN conversion (in-place on a deep copy)."""
    model = copy.deepcopy(ann_model)
    thr_list = list(thresholds.values())
    layer_idx = 0; prev_thr = 1.0
    for _, m in model.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)) and layer_idx < len(thr_list):
            cur = thr_list[layer_idx]
            with torch.no_grad():
                m.weight.data *= prev_thr / cur
                if m.bias is not None: m.bias.data /= cur
            prev_thr = cur; layer_idx += 1
    return model

print('Threshold balancing & converter defined.')

### SNN Model Wrapper

In [ ]:
class SNNModel(nn.Module):
    """Wraps a normalized ANN as a multi-timestep SNN (rate coding, IF neurons)."""
    def __init__(self, ann_model, thresholds, max_timestep=32):
        super().__init__()
        self.max_timestep = max_timestep
        thr_list = list(thresholds.values())
        self.spiking_layers = nn.ModuleList()
        thr_idx = 0
        for _, m in ann_model.named_modules():
            if isinstance(m, nn.Conv2d):
                thr = thr_list[thr_idx] if thr_idx < len(thr_list) else 1.0
                sc = SpikingConv2d(m.in_channels, m.out_channels,
                                   m.kernel_size[0], m.stride[0], m.padding[0],
                                   m.bias is not None, threshold=thr)
                sc.conv.weight.data = m.weight.data.clone()
                if m.bias is not None: sc.conv.bias.data = m.bias.data.clone()
                self.spiking_layers.append(sc); thr_idx += 1
            elif isinstance(m, nn.MaxPool2d):
                self.spiking_layers.append(nn.MaxPool2d(m.kernel_size, m.stride))
            elif isinstance(m, nn.AdaptiveAvgPool2d):
                sz = m.output_size
                self.spiking_layers.append(nn.AdaptiveAvgPool2d(sz if isinstance(sz,tuple) else (sz,sz)))
            elif isinstance(m, nn.BatchNorm2d):
                sb = SpikingBN(m.num_features); sb.bn.load_state_dict(m.state_dict())
                self.spiking_layers.append(sb)
        last_lin = None
        for m in ann_model.modules():
            if isinstance(m, nn.Linear): last_lin = m
        self.classifier = nn.Linear(last_lin.in_features, last_lin.out_features,
                                    bias=last_lin.bias is not None)
        self.classifier.weight.data = last_lin.weight.data.clone()
        if last_lin.bias is not None: self.classifier.bias.data = last_lin.bias.data.clone()

    def reset(self):
        for l in self.spiking_layers:
            if hasattr(l, 'reset'): l.reset()

    def _one_step(self, x):
        for l in self.spiking_layers: x = l(x)
        return self.classifier(x.view(x.size(0),-1))

    def forward(self, x, timesteps=None):
        T = timesteps or self.max_timestep; self.reset()
        out = None
        for _ in range(T):
            o = self._one_step(x)
            if out is None: out = o
            else: out.add_(o)  # in-place accumulation
        return out / T

    def forward_at_timesteps(self, x, checkpoints):
        self.reset(); out_sum = None; outputs = {}; max_t = max(checkpoints)
        checkpoints_set = set(checkpoints)
        for t in range(1, max_t+1):
            o = self._one_step(x)
            if out_sum is None:
                out_sum = o
            else:
                out_sum.add_(o)  # in-place: avoids allocating a new tensor each step
            if t in checkpoints_set:
                outputs[t] = out_sum / t  # only materialise at checkpoints
        return outputs

print('SNNModel defined.')

### Multi-Exit SNN

In [ ]:
class ExitBranch(nn.Module):
    def __init__(self, in_features, num_classes, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden), nn.ReLU(True), nn.Dropout(0.1),
            nn.Linear(hidden, num_classes))
    def forward(self, x): return self.net(x)

class MultiExitSNN(nn.Module):
    def __init__(self, snn_model, exit_timesteps, num_classes, hidden=128):
        super().__init__()
        self.snn = snn_model
        self.exit_timesteps = sorted(exit_timesteps)
        self.num_classes = num_classes
        self.max_t = max(exit_timesteps)
        feat_dim = snn_model.classifier.in_features
        self.exits = nn.ModuleDict({str(t): ExitBranch(feat_dim, num_classes, hidden)
                                    for t in self.exit_timesteps})

    def forward(self, x, target_t=None):
        self.snn.reset(); spike_sum = None; outputs = {}
        run_to = (target_t if (target_t and not self.training) else self.max_t)
        exit_set = set(self.exit_timesteps)
        for t in range(1, run_to+1):
            sp = x
            for layer in self.snn.spiking_layers: sp = layer(sp)
            flat = sp.view(sp.size(0),-1)
            if spike_sum is None: spike_sum = flat
            else: spike_sum.add_(flat)  # in-place
            if t in exit_set:
                feat = spike_sum / t
                outputs[t] = self.exits[str(t)](feat)
                if not self.training and target_t and t >= target_t: break
        return outputs

    def forward_single_exit(self, x, timestep):
        exit_t = next((t for t in self.exit_timesteps if t >= timestep), self.exit_timesteps[-1])
        return self.forward(x, exit_t)[exit_t]

print('MultiExitSNN defined.')

### Power-Law Curve Fitting

In [ ]:
def power_law(t, alpha, beta, gamma): return alpha * np.power(t, beta) + gamma

def fit_single(timesteps, perfs):
    ts = np.asarray(timesteps, float); ps = np.asarray(perfs, float)
    valid = np.isfinite(ps) & (ts > 0)
    if valid.sum() < 3: return 0.3, 0.5, ps.mean()
    ts, ps = ts[valid], ps[valid]
    for p0 in [(0.1,0.5,ps.min()),(0.5,0.3,0.0),(1.0,0.7,ps[0])]:
        try:
            params, _ = curve_fit(power_law, ts, ps, p0=p0,
                                  bounds=([0,0.01,-1],[10,2,1]), maxfev=3000)
            return float(np.clip(params[0],1e-6,None)),                    float(np.clip(params[1],0.01,1.5)),                    float(np.clip(params[2],-0.5,1.0))
        except: pass
    g = ps[0]; a = max((ps[-1]-ps[0])/(ts[-1]**0.5), 1e-6)
    return a, 0.5, g

def batch_fit(timesteps, all_perfs):
    N = all_perfs.shape[0]
    As, Bs, Gs = np.zeros(N), np.zeros(N), np.zeros(N)
    for i in range(N): As[i],Bs[i],Gs[i] = fit_single(timesteps, all_perfs[i])
    return As, Bs, Gs

def optimal_t(alpha, beta, gamma, target, max_t=32):
    if gamma >= target: return 1
    if alpha < 1e-6: return max_t
    num = target - gamma
    if num <= 0: return 1
    t = int(np.ceil((num/alpha)**(1.0/max(beta,0.01))))
    return max(1, min(t, max_t))

print('Curve fitting defined.')

### Complexity Predictor (lightweight CNN)

In [ ]:
class ComplexityPredictor(nn.Module):
    """Lightweight CNN: image → (α, β, γ) scaling law parameters."""
    def __init__(self, in_channels=3, hidden_dims=[64,128,64]):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels,32,3,1,1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,1,1),          nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,1,1),         nn.BatchNorm2d(128),nn.ReLU(True),
            nn.AdaptiveAvgPool2d((1,1)),
        )
        layers = []; in_d = 128
        for h in hidden_dims:
            layers += [nn.Linear(in_d,h), nn.ReLU(True), nn.Dropout(0.1)]; in_d = h
        layers.append(nn.Linear(in_d, 3))
        self.head = nn.Sequential(*layers)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif isinstance(m, nn.Linear): nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        f = self.features(x).flatten(1)
        raw = self.head(f)
        return {
            'alpha': F.softplus(raw[:,0]),
            'beta' : torch.sigmoid(raw[:,1]),
            'gamma': torch.sigmoid(raw[:,2]),
        }

    def predict_t(self, x, target=0.90, exit_pts=None, max_t=32):
        p = self.forward(x)
        a,b,g = p['alpha'], p['beta'], p['gamma']
        num = (target - g).clamp(min=1e-6)
        t = (num / a.clamp(min=1e-6)).clamp(min=1e-6).pow(1.0/b.clamp(min=0.01))
        t = t.clamp(1, max_t)
        if exit_pts:
            exits = torch.tensor(exit_pts, dtype=t.dtype, device=t.device)
            valid = exits.unsqueeze(0) >= t.unsqueeze(1)
            max_e = exits[-1]
            t = exits.unsqueeze(0).masked_fill(~valid, max_e+1).min(1).values.clamp(max=max_e)
        return t, p

print('ComplexityPredictor defined.')

---
## Phase 1 — ANN Pretraining
Train ResNet-20 on CIFAR-10 for 30 epochs with cosine LR schedule.
Expected time: **~10–12 min** on T4.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train(); loss_sum = corr = n = 0
    for imgs, tgts in loader:
        imgs, tgts = imgs.to(device), tgts.to(device)
        optimizer.zero_grad()
        out = model(imgs); loss = criterion(out, tgts)
        loss.backward(); optimizer.step()
        loss_sum += loss.item()*imgs.size(0)
        corr += out.argmax(1).eq(tgts).sum().item(); n += imgs.size(0)
    return loss_sum/n, 100.*corr/n

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval(); loss_sum = corr = n = 0
    for imgs, tgts in loader:
        imgs, tgts = imgs.to(device), tgts.to(device)
        out = model(imgs); loss = criterion(out, tgts)
        loss_sum += loss.item()*imgs.size(0)
        corr += out.argmax(1).eq(tgts).sum().item(); n += imgs.size(0)
    return loss_sum/n, 100.*corr/n

print('Training helpers defined.')

In [ ]:
import gc
gc.collect()
if device.type == 'cuda': torch.cuda.empty_cache()
t0 = time.time()
mc = CFG['model']

train_loader, test_loader = get_cifar10_loaders(DATA, batch_size=mc['batch_size'])
ann = resnet20(CFG['dataset']['num_classes']).to(device)

optimizer = optim.SGD(ann.parameters(), lr=mc['lr'],
                      momentum=mc['momentum'], weight_decay=mc['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=mc['epochs'])
criterion = nn.CrossEntropyLoss()

best_acc = 0.0; p1_history = []
print(f"Training ResNet-20 for {mc['epochs']} epochs...")
pbar = tqdm(range(mc['epochs']), desc='Phase 1')
for epoch in pbar:
    tr_loss, tr_acc = train_one_epoch(ann, train_loader, criterion, optimizer, device)
    te_loss, te_acc = evaluate(ann, test_loader, criterion, device)
    scheduler.step()
    p1_history.append({'epoch':epoch+1,'tr_acc':tr_acc,'te_acc':te_acc,'tr_loss':tr_loss})
    pbar.set_postfix({'train':f'{tr_acc:.1f}%','test':f'{te_acc:.1f}%'})
    if te_acc > best_acc:
        best_acc = te_acc
        torch.save({'model_state_dict':ann.state_dict(),'best_acc':best_acc},
                   CKPT/'ann_best.pth')

ann.load_state_dict(torch.load(CKPT/'ann_best.pth')['model_state_dict'])
print(f'\nPhase 1 done | Best test acc: {best_acc:.2f}% | Time: {(time.time()-t0)/60:.1f} min')

In [ ]:
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(12,4))
epochs_plot = [r['epoch'] for r in p1_history]
ax1.plot(epochs_plot,[r['tr_acc'] for r in p1_history], label='Train')
ax1.plot(epochs_plot,[r['te_acc'] for r in p1_history], label='Test')
ax1.set(xlabel='Epoch', ylabel='Accuracy (%)', title='Phase 1: ANN Training Accuracy'); ax1.legend()
ax2.plot(epochs_plot,[r['tr_loss'] for r in p1_history])
ax2.set(xlabel='Epoch', ylabel='Loss', title='Phase 1: Training Loss')
plt.tight_layout(); plt.savefig(PLOT/'phase1_training.png',dpi=120,bbox_inches='tight')
show_plot(plt.gcf(), PLOT/'phase1_training.png')
print(f'Plot saved to {PLOT}/phase1_training.png')

---
## Phase 2 — SNN Profiling & Curve Fitting
Convert the ANN to SNN, run each sample at timesteps [1,2,4,8,16,32],
then fit per-sample power-law curves (α, β, γ).
Expected time: **~8–12 min** on T4 (5 000-sample subset).

In [ ]:
t0 = time.time()
sc = CFG['snn']; cc = CFG['conversion']; pc = CFG['profiling']

# ── 1. Threshold balancing ───────────────────────────────────────────────────
print('Computing thresholds...')
calib_loader = get_cifar10_loaders(DATA, calib_only=True, calib_n=cc['calibration_samples'])
balancer = ThresholdBalancer(percentile=cc['percentile'])
thresholds = balancer.compute_thresholds(ann, calib_loader, device)
print(f'  {len(thresholds)} thresholds computed.')

# ── 2. Build SNN ─────────────────────────────────────────────────────────────
print('Building SNN...')
# Free calibration activations before building SNN
del balancer
gc.collect()
if device.type == 'cuda': torch.cuda.empty_cache()
ann_norm = convert_ann_to_snn_weights(ann, thresholds)
snn = SNNModel(ann_norm, thresholds, max_timestep=sc['max_timestep']).to(device)
snn.eval()
torch.save({'snn_state_dict':snn.state_dict(),'thresholds':list(thresholds.values())},
           CKPT/'snn_converted.pth')
del ann_norm  # free the CPU copy of the normalised weights
gc.collect()

# ── 3. Profile subset of training data (streaming — never store full perfs) ──
print(f'Profiling {pc["num_samples"]} samples at timesteps {sc["timesteps"]}...')
mean_n = [0.4914,0.4822,0.4465]; std_n = [0.2023,0.1994,0.2010]
base_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean_n,std_n)])
full_ds  = torchvision.datasets.CIFAR10(str(DATA), train=True, download=False, transform=base_tf)
idx_sub  = torch.randperm(len(full_ds))[:pc['num_samples']]
sub_ds   = Subset(full_ds, idx_sub)
sub_loader = DataLoader(sub_ds, batch_size=pc['batch_size'], shuffle=False,
                        num_workers=2, pin_memory=(device.type=='cuda'))

# Pre-allocate result arrays — avoid growing lists of arrays
N_prof = len(sub_ds); T_cnt = len(sc['timesteps'])
alphas = np.zeros(N_prof, dtype=np.float32)
betas  = np.zeros(N_prof, dtype=np.float32)
gammas = np.zeros(N_prof, dtype=np.float32)
labels = np.zeros(N_prof, dtype=np.int64)
# Keep only per-timestep mean accuracy (not per-sample) to avoid big array
mean_acc_per_t = np.zeros(T_cnt, dtype=np.float64)
ts_arr = np.array(sc['timesteps'], float)

# ── 4. Stream-profile + fit-on-the-fly (no full perfs matrix in RAM) ─────────
print('Fitting power-law curves per batch (streaming)...')
write_ptr = 0
with torch.no_grad():
    for imgs, tgts in tqdm(sub_loader, desc='Profiling'):
        imgs, tgts = imgs.to(device), tgts.to(device)
        outs = snn.forward_at_timesteps(imgs, sc['timesteps'])
        B = imgs.size(0)

        # Build (B, T) correctness matrix — float32 on CPU only
        perf_batch = np.stack(
            [outs[t].argmax(1).eq(tgts).float().cpu().numpy() for t in sc['timesteps']],
            axis=1
        )  # shape (B, T) — discarded after this batch

        # Cumulative-max smoothing per sample
        perf_smooth = np.maximum.accumulate(perf_batch, axis=1)
        perf_smooth = np.cumsum(perf_smooth, axis=1) / np.arange(1, T_cnt+1)

        # Accumulate running mean accuracy per timestep
        mean_acc_per_t += perf_smooth.sum(axis=0)

        # Fit curve per sample — results go straight into pre-allocated arrays
        for i in range(B):
            a, b, g = fit_single(ts_arr, perf_smooth[i])
            alphas[write_ptr+i] = a
            betas[write_ptr+i]  = b
            gammas[write_ptr+i] = g
        labels[write_ptr:write_ptr+B] = tgts.cpu().numpy()
        write_ptr += B

        # Explicitly release intermediates
        del imgs, tgts, outs, perf_batch, perf_smooth
        if device.type == 'cuda': torch.cuda.empty_cache()

mean_acc_per_t /= N_prof  # normalise to mean

print(f'  α: {alphas.mean():.4f}±{alphas.std():.4f}')
print(f'  β: {betas.mean():.4f}±{betas.std():.4f}')
print(f'  γ: {gammas.mean():.4f}±{gammas.std():.4f}')

# Save — note: no perfs_smooth matrix saved (too big); only parameters
np.savez(CKPT/'profiling_results.npz',
         alphas=alphas, betas=betas, gammas=gammas,
         mean_acc_per_t=mean_acc_per_t, timesteps=ts_arr, labels=labels)

# Free calibration data, no longer needed
del calib_loader, ann_norm, snn
gc.collect()
if device.type == 'cuda': torch.cuda.empty_cache()
print(f'\nPhase 2 done | Time: {(time.time()-t0)/60:.1f} min')

In [ ]:
t_opts = np.array([optimal_t(a,b,g, CFG['scaling_law']['target_accuracy'],
                              CFG['snn']['max_timestep'])
                   for a,b,g in zip(alphas,betas,gammas)])

fig, axes = plt.subplots(1,3,figsize=(15,4))

# Difficulty histogram
axes[0].hist(t_opts, bins=len(CFG['snn']['exit_points']),
             edgecolor='black', color='steelblue')
for t in CFG['snn']['exit_points']:
    axes[0].axvline(t, color='red', linestyle='--', alpha=0.5)
axes[0].set(xlabel='Optimal Timestep T*', ylabel='Count',
            title='Sample Difficulty Distribution')

# α & β scatter
sc_plot = axes[1].scatter(alphas, betas, c=t_opts, cmap='RdYlGn_r', alpha=0.4, s=8)
plt.colorbar(sc_plot, ax=axes[1], label='T*')
axes[1].set(xlabel='α (amplitude)', ylabel='β (exponent)', title='Scaling Law Parameters')

# Mean accuracy vs timestep — use pre-computed mean (no big array needed)
axes[2].plot(CFG['snn']['timesteps'], mean_acc_per_t*100, 'o-', color='steelblue')
axes[2].set(xlabel='Timestep T', ylabel='Mean Accuracy (%)',
            title='SNN Accuracy vs Timestep')
axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.savefig(PLOT/'phase2_analysis.png',dpi=120,bbox_inches='tight')
show_plot(plt.gcf(), PLOT/'phase2_analysis.png')

# Show 6 example scaling law fits using only fitted parameters (no perfs_smooth needed)
fig, axes = plt.subplots(2,3,figsize=(12,6))
idxs = np.random.choice(len(alphas),6,replace=False)
ts_dense = np.linspace(1,CFG['snn']['max_timestep'],100)
for ax,i in zip(axes.flat, idxs):
    fit_curve = power_law(ts_dense, alphas[i], betas[i], gammas[i])
    ax.plot(ts_dense, fit_curve*100, 'r-', label=f'fit α={alphas[i]:.2f} β={betas[i]:.2f}')
    ax.axhline(CFG['scaling_law']['target_accuracy']*100, color='green', linestyle='--', label='target')
    ax.set(xlabel='T', ylabel='Pred Acc %', title=f'Sample {i} (T*={t_opts[i]})'); ax.legend(fontsize=6)
plt.tight_layout(); plt.savefig(PLOT/'phase2_sample_fits.png',dpi=120,bbox_inches='tight')
show_plot(plt.gcf(), PLOT/'phase2_sample_fits.png')
print('Phase 2 plots saved.')

---
## Phase 3 — Joint Training
Train the **Complexity Predictor** (image → α,β,γ) and the **Multi-Exit SNN** branches simultaneously.
The SNN backbone is frozen; only the exit classifiers and predictor are trained.
Expected time: **~8–10 min** on T4 (20 epochs).

In [ ]:
def compute_t_optimal_batch(alphas, betas, gammas, target, max_t):
    """Differentiable T* computation for a batch."""
    num = (target - gammas).clamp(min=1e-6)
    t = (num / alphas.clamp(min=1e-6)).clamp(min=1e-6).pow(1.0/betas.clamp(min=0.01))
    return t.clamp(1, max_t)

def snap_to_exits(t, exit_pts_tensor):
    valid = exit_pts_tensor.unsqueeze(0) >= t.unsqueeze(1)
    max_e = exit_pts_tensor[-1]
    return exit_pts_tensor.unsqueeze(0).masked_fill(~valid,max_e+1).min(1).values.clamp(max=max_e)

print('Phase 3 loss helpers defined.')

In [ ]:
import gc
t0 = time.time()
pred_cfg = CFG['predictor']; sc_cfg = CFG['snn']; sl_cfg = CFG['scaling_law']
inf_cfg  = CFG['inference']

# ── Memory cleanup before Phase 3 ────────────────────────────────────────────
gc.collect()
if device.type == 'cuda': torch.cuda.empty_cache()

# ── Load profiling results ───────────────────────────────────────────────────
prof = np.load(CKPT/'profiling_results.npz')
# float16 on CPU — halves RAM, precision is fine for MSE supervision
tgt_alpha = torch.tensor(prof['alphas'], dtype=torch.float16)
tgt_beta  = torch.tensor(prof['betas'],  dtype=torch.float16)
tgt_gamma = torch.tensor(prof['gammas'], dtype=torch.float16)
del prof; gc.collect()

# ── Rebuild SNN from checkpoint ──────────────────────────────────────────────
snn_ckpt = torch.load(CKPT/'snn_converted.pth', map_location=device)
ann_norm2 = convert_ann_to_snn_weights(ann, thresholds)
snn2 = SNNModel(ann_norm2, thresholds, max_timestep=sc_cfg['max_timestep']).to(device)
snn2.load_state_dict(snn_ckpt['snn_state_dict'])
del ann_norm2  # free CPU copy immediately after weights transferred to GPU
gc.collect()

# ── Build Multi-Exit SNN ─────────────────────────────────────────────────────
me_snn = MultiExitSNN(snn2, sc_cfg['exit_points'], CFG['dataset']['num_classes']).to(device)
for p in me_snn.snn.parameters(): p.requires_grad_(False)  # freeze backbone

# ── Build Complexity Predictor ────────────────────────────────────────────────
predictor = ComplexityPredictor(hidden_dims=pred_cfg['hidden_dims']).to(device)

# ── Optimizers ───────────────────────────────────────────────────────────────
pred_opt = optim.Adam(predictor.parameters(), lr=pred_cfg['lr'], weight_decay=1e-4)
exit_opt = optim.Adam(me_snn.exits.parameters(), lr=pred_cfg['lr'], weight_decay=1e-4)
pred_sch  = optim.lr_scheduler.CosineAnnealingLR(pred_opt, T_max=pred_cfg['epochs'])
exit_sch  = optim.lr_scheduler.CosineAnnealingLR(exit_opt, T_max=pred_cfg['epochs'])
ce_loss   = nn.CrossEntropyLoss()

# ── Build profiling dataset (images + labels from the profiled subset) ────────
mean_n = [0.4914,0.4822,0.4465]; std_n=[0.2023,0.1994,0.2010]
base_tf2 = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean_n,std_n)])
full_ds2  = torchvision.datasets.CIFAR10(str(DATA), train=True, download=False, transform=base_tf2)
prof_ds   = Subset(full_ds2, idx_sub.tolist())
prof_loader = DataLoader(prof_ds, batch_size=pred_cfg['batch_size'], shuffle=True,
                         num_workers=2, pin_memory=(device.type=='cuda'), drop_last=True)

exit_pts_t = torch.tensor(sc_cfg['exit_points'], dtype=torch.float32, device=device)
energy_w   = inf_cfg['energy_weight']
target_acc = sl_cfg['target_accuracy']
max_t      = sc_cfg['max_timestep']

best_metric = 0.0; p3_history = []
print(f'Joint training for {pred_cfg["epochs"]} epochs...')
pbar3 = tqdm(range(pred_cfg['epochs']), desc='Phase 3')

for epoch in pbar3:
    me_snn.train(); predictor.train()
    ep_pred_loss = ep_exit_loss = ep_t_pred = n_batches = 0
    exit_correct = {t:0 for t in sc_cfg['exit_points']}; exit_total = {t:0 for t in sc_cfg['exit_points']}

    for batch_idx, (imgs, tgts) in enumerate(prof_loader):
        imgs, tgts = imgs.to(device), tgts.to(device)
        # Map batch to profiling labels by sequential index
        start = batch_idx * pred_cfg['batch_size']
        end   = start + imgs.size(0)
        ta = tgt_alpha[start:end].to(device).float()
        tb = tgt_beta[start:end].to(device).float()
        tg = tgt_gamma[start:end].to(device).float()

        # ── Predictor loss ───────────────────────────────────────────────────
        pred_opt.zero_grad()
        params = predictor(imgs)
        pa, pb, pg = params['alpha'], params['beta'], params['gamma']
        param_loss = ((pa-ta).pow(2).mean() + (pb-tb).pow(2).mean() + (pg-tg).pow(2).mean())/3
        pred_t  = compute_t_optimal_batch(pa, pb, pg, target_acc, max_t)
        tgt_t_v = compute_t_optimal_batch(ta, tb, tg, target_acc, max_t)
        t_loss  = (pred_t.log()-tgt_t_v.log().detach()).pow(2).mean()
        e_loss  = (pred_t / max_t).mean()
        total_pred = param_loss + t_loss + energy_w * e_loss
        total_pred.backward(); pred_opt.step()

        # ── Exit branch loss ──────────────────────────────────────────────────
        exit_opt.zero_grad()
        exit_outs = me_snn(imgs)
        ex_loss = sum(ce_loss(exit_outs[t], tgts) for t in sc_cfg['exit_points'])
        ex_loss.backward(); exit_opt.step()

        ep_pred_loss += total_pred.item(); ep_exit_loss += ex_loss.item()
        ep_t_pred    += pred_t.mean().item(); n_batches += 1

        for t in sc_cfg['exit_points']:
            preds = exit_outs[t].argmax(1)
            exit_correct[t] += preds.eq(tgts).sum().item()
            exit_total[t]   += tgts.size(0)

        # Free batch tensors each step — critical for GPU memory
        del imgs, tgts, params, pa, pb, pg, ta, tb, tg
        del total_pred, ex_loss, exit_outs, pred_t
        if device.type == 'cuda' and n_batches % 20 == 0:
            torch.cuda.empty_cache()

    pred_sch.step(); exit_sch.step()
    avg_t = ep_t_pred / n_batches
    exit_accs = {t: 100.*exit_correct[t]/exit_total[t] for t in sc_cfg['exit_points']}
    final_acc  = exit_accs[sc_cfg['exit_points'][-1]]
    energy_sav = 1.0 - avg_t/max_t
    metric = final_acc*0.7 + energy_sav*100*0.3
    p3_history.append({'epoch':epoch+1,'pred_loss':ep_pred_loss/n_batches,
                        'exit_loss':ep_exit_loss/n_batches,'avg_t':avg_t,
                        'exit_accs':exit_accs,'energy_sav':energy_sav})
    pbar3.set_postfix({'pred':f'{ep_pred_loss/n_batches:.3f}',
                       'exit':f'{ep_exit_loss/n_batches:.3f}',
                       f'acc@T{sc_cfg["exit_points"][-1]}':f'{final_acc:.1f}%',
                       'avg_T':f'{avg_t:.1f}'})
    if metric > best_metric:
        best_metric = metric
        torch.save({'exit_branches_state_dict': me_snn.exits.state_dict(),
                    'predictor_state_dict':     predictor.state_dict(),
                    'epoch': epoch, 'metric': metric}, CKPT/'phase3_best.pth')

# Load best weights
ckpt3 = torch.load(CKPT/'phase3_best.pth', map_location=device)
me_snn.exits.load_state_dict(ckpt3['exit_branches_state_dict'])
predictor.load_state_dict(ckpt3['predictor_state_dict'])
del tgt_alpha, tgt_beta, tgt_gamma, ann_norm2, snn2, snn_ckpt, ckpt3
gc.collect()
if device.type == 'cuda': torch.cuda.empty_cache()
print(f'\nPhase 3 done | Best metric: {best_metric:.2f} | Time: {(time.time()-t0)/60:.1f} min')

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,4))
ep3 = [r['epoch'] for r in p3_history]
axes[0].plot(ep3,[r['pred_loss'] for r in p3_history],label='Predictor')
axes[0].plot(ep3,[r['exit_loss'] for r in p3_history],label='Exit CE')
axes[0].set(xlabel='Epoch',ylabel='Loss',title='Phase 3 Losses'); axes[0].legend()

for t in CFG['snn']['exit_points']:
    axes[1].plot(ep3,[r['exit_accs'][t] for r in p3_history],label=f'T={t}')
axes[1].set(xlabel='Epoch',ylabel='Accuracy (%)',title='Exit Branch Accuracy'); axes[1].legend()

axes[2].plot(ep3,[r['avg_t'] for r in p3_history],'b-',label='Avg T used')
ax2r = axes[2].twinx()
ax2r.plot(ep3,[r['energy_sav']*100 for r in p3_history],'r--',label='Energy savings %')
axes[2].set(xlabel='Epoch',ylabel='Avg Timestep',title='Efficiency vs Epoch')
axes[2].legend(loc='upper left'); ax2r.legend(loc='upper right'); ax2r.set_ylabel('Energy Savings %')

plt.tight_layout(); plt.savefig(PLOT/'phase3_training.png',dpi=120,bbox_inches='tight')
show_plot(plt.gcf(), PLOT/'phase3_training.png')
print('Phase 3 plot saved.')

---
## Evaluation — Adaptive Inference
Run the full NeuroScale++ system on the CIFAR-10 test set and measure:
- **Accuracy** at each exit point vs. the ANN baseline
- **Average timesteps used** per sample
- **Energy savings** (1 − avg_T / max_T)
- **Per-class accuracy**

In [ ]:
t0 = time.time()
me_snn.eval(); predictor.eval()

# Use numpy arrays pre-allocated to test set size to avoid list growth
N_test = len(test_loader.dataset)
preds_arr  = np.zeros(N_test, dtype=np.int64)
tgts_arr   = np.zeros(N_test, dtype=np.int64)
T_arr      = np.zeros(N_test, dtype=np.int32)
conf_arr   = np.zeros(N_test, dtype=np.float32)
write_idx  = 0

exit_pts = CFG['snn']['exit_points']
exit_pts_tensor = torch.tensor(exit_pts, dtype=torch.float32, device=device)

with torch.no_grad():
    for imgs, tgts in tqdm(test_loader, desc='Evaluating'):
        imgs = imgs.to(device)
        B = imgs.size(0)

        # Predict optimal timestep per sample
        params = predictor(imgs)
        pa,pb,pg = params['alpha'], params['beta'], params['gamma']
        t_cont  = compute_t_optimal_batch(pa, pb, pg,
                                          CFG['scaling_law']['target_accuracy'],
                                          CFG['snn']['max_timestep'])
        t_snap  = snap_to_exits(t_cont, exit_pts_tensor).int()

        # Group by exit point — run only once per exit group
        preds_batch = torch.zeros(B, dtype=torch.long, device=device)
        confs_batch = torch.zeros(B, device=device)
        for et in exit_pts:
            mask = (t_snap == et)
            if not mask.any(): continue
            logits = me_snn.forward_single_exit(imgs[mask], et)
            probs  = torch.softmax(logits, 1)
            conf, pred = probs.max(1)
            preds_batch[mask] = pred
            confs_batch[mask] = conf
            del logits, probs  # free immediately

        # Write directly into pre-allocated arrays
        end_idx = write_idx + B
        preds_arr[write_idx:end_idx] = preds_batch.cpu().numpy()
        tgts_arr[write_idx:end_idx]  = tgts.numpy()
        T_arr[write_idx:end_idx]     = t_snap.cpu().numpy()
        conf_arr[write_idx:end_idx]  = confs_batch.cpu().numpy()
        write_idx = end_idx

        del imgs, params, pa, pb, pg, t_cont, t_snap, preds_batch, confs_batch

preds  = preds_arr[:write_idx]
targets= tgts_arr[:write_idx]
T_used = T_arr[:write_idx]
confs  = conf_arr[:write_idx]

correct = (preds == targets)
total_acc = correct.mean()*100
avg_T     = T_used.mean()
max_T     = CFG['snn']['max_timestep']
energy_sav= (1 - avg_T/max_T)*100

# ANN baseline accuracy (already computed)
_, ann_base_acc = evaluate(ann, test_loader, criterion, device)

print('='*55)
print(f'  ANN baseline accuracy  : {ann_base_acc:.2f}%')
print(f'  NeuroScale++ accuracy  : {total_acc:.2f}%')
print(f'  Accuracy drop          : {ann_base_acc-total_acc:.2f}%')
print(f'  Avg timesteps used     : {avg_T:.2f} / {max_T}')
print(f'  Energy savings         : {energy_sav:.1f}%')
print(f'  Speedup factor         : {max_T/avg_T:.2f}x')
print('='*55)
print(f'\nEvaluation done | Time: {(time.time()-t0)/60:.1f} min')

In [ ]:
print('Per-exit breakdown:')
for et in exit_pts:
    mask = T_used == et
    n = mask.sum()
    if n > 0:
        acc = correct[mask].mean()*100
        pct = n/len(T_used)*100
        print(f'  T={et:2d}: {n:5d} samples ({pct:5.1f}%) | acc={acc:.2f}%')

In [ ]:
fig = plt.figure(figsize=(16,10))
gs  = fig.add_gridspec(2,3,hspace=0.4,wspace=0.35)

# 1. Exit distribution pie chart
ax1 = fig.add_subplot(gs[0,0])
counts = [int((T_used==et).sum()) for et in exit_pts]
ax1.pie(counts, labels=[f'T={t}' for t in exit_pts], autopct='%1.1f%%',
        colors=plt.cm.Set2.colors[:len(exit_pts)])
ax1.set_title('Sample Exit Distribution')

# 2. Per-exit accuracy vs baseline
ax2 = fig.add_subplot(gs[0,1])
exit_accs_eval = []
for et in exit_pts:
    mask = T_used == et
    exit_accs_eval.append(correct[mask].mean()*100 if mask.sum()>0 else 0)
bars = ax2.bar([str(t) for t in exit_pts], exit_accs_eval, color='steelblue', label='NeuroScale++')
ax2.axhline(ann_base_acc, color='red', linestyle='--', linewidth=2, label=f'ANN baseline ({ann_base_acc:.1f}%)')
ax2.set(xlabel='Exit Timestep', ylabel='Accuracy (%)', title='Accuracy by Exit Point',
        ylim=(max(0,min(exit_accs_eval)-10),100)); ax2.legend()
for bar,acc in zip(bars,exit_accs_eval):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{acc:.1f}%',
             ha='center',va='bottom',fontsize=8)

# 3. Confidence histogram
ax3 = fig.add_subplot(gs[0,2])
ax3.hist(confs, bins=50, color='green', alpha=0.7, edgecolor='black')
ax3.set(xlabel='Confidence', ylabel='Count', title='Prediction Confidence Dist.')

# 4. Per-class accuracy
ax4 = fig.add_subplot(gs[1,:2])
classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
class_acc = []
for c in range(10):
    mask = targets==c
    class_acc.append(correct[mask].mean()*100 if mask.sum()>0 else 0)
colors_bar = ['green' if a>=ann_base_acc else 'orange' if a>=ann_base_acc-5 else 'red' for a in class_acc]
ax4.bar(classes, class_acc, color=colors_bar)
ax4.axhline(ann_base_acc, color='red', linestyle='--', label=f'Baseline {ann_base_acc:.1f}%')
ax4.set(xlabel='Class', ylabel='Accuracy (%)', title='Per-Class Accuracy'); ax4.legend()
plt.xticks(rotation=30, ha='right')

# 5. Energy savings summary text
ax5 = fig.add_subplot(gs[1,2])
ax5.axis('off')
summary = (
    f'NEUROSCALE++ SUMMARY\n'
    f'{"="*28}\n'
    f'ANN Accuracy:     {ann_base_acc:.2f}%\n'
    f'SNN Accuracy:     {total_acc:.2f}%\n'
    f'Acc Drop:         {ann_base_acc-total_acc:.2f}%\n'
    f'Avg Timesteps:    {avg_T:.1f} / {max_T}\n'
    f'Energy Savings:   {energy_sav:.1f}%\n'
    f'Speedup:          {max_T/avg_T:.2f}x\n'
    f'Samples tested:   {len(preds)}'
)
ax5.text(0.05,0.95,summary,transform=ax5.transAxes,fontsize=11,
         verticalalignment='top',fontfamily='monospace',
         bbox=dict(boxstyle='round',facecolor='lightyellow',alpha=0.8))

plt.suptitle('NeuroScale++ Evaluation Results', fontsize=14, fontweight='bold', y=1.01)
plt.savefig(PLOT/'evaluation_results.png',dpi=120,bbox_inches='tight')
show_plot(plt.gcf(), PLOT/'evaluation_results.png')
print('Evaluation plots saved.')

In [ ]:
# Show how the predictor compares to ground-truth scaling law fits on test samples
fig, axes = plt.subplots(2,4,figsize=(16,7))

# Get a batch of test images
test_imgs, test_tgts = next(iter(test_loader))
test_imgs = test_imgs.to(device)[:8]

with torch.no_grad():
    pred_params = predictor(test_imgs)

ts_dense = np.linspace(1, CFG['snn']['max_timestep'], 100)
for i, ax in enumerate(axes.flat):
    pa = pred_params['alpha'][i].item()
    pb = pred_params['beta'][i].item()
    pg = pred_params['gamma'][i].item()
    curve = power_law(ts_dense, pa, pb, pg)
    t_star = optimal_t(pa,pb,pg, CFG['scaling_law']['target_accuracy'], CFG['snn']['max_timestep'])
    ax.plot(ts_dense, curve*100, 'b-', linewidth=2)
    ax.axhline(CFG['scaling_law']['target_accuracy']*100, color='green', linestyle='--', alpha=0.7)
    ax.axvline(t_star, color='red', linestyle=':', alpha=0.8)
    ax.set(xlabel='T', ylabel='Pred Acc %',
           title=f'α={pa:.2f} β={pb:.2f}\nT*={t_star}')
    ax.set_ylim(0,100); ax.grid(True,alpha=0.3)

plt.suptitle('Predicted Scaling Laws for 8 Test Samples\n(red dashed = T*, green = target accuracy)',
             fontsize=12)
plt.tight_layout(); plt.savefig(PLOT/'scaling_law_examples.png',dpi=120,bbox_inches='tight'); plt.show()

## Download Results
Save all plots and checkpoints as a zip file.

In [ ]:
import shutil
shutil.make_archive(str(BASE / 'neuroscale_results'), 'zip', str(BASE), 'plots')
print(f'Results zipped to: {BASE}/neuroscale_results.zip')

try:
    from google.colab import files
    files.download(str(BASE / 'neuroscale_results.zip'))
    print('Downloading results zip...')
except ImportError:
    print(f'Running locally — open your plots at:')
    print(f'  {PLOT.resolve()}')
    import subprocess
    subprocess.run(['open', str(PLOT)], check=False)  # opens Finder on Mac

print('\nAll plots:')
for f in sorted(PLOT.glob('*.png')): print(f'  {f.name}')

---
## Tips for Faster Runs

| Parameter | Default | Speed-up option |
|---|---|---|
| `model.epochs` | 30 | 15 (loses ~3% acc) |
| `profiling.num_samples` | 5 000 | 2 000 |
| `predictor.epochs` | 20 | 10 |
| `snn.max_timestep` | 32 | 16 |

**For a research-quality run:** increase `model.epochs` to 100, `profiling.num_samples` to 20 000,
`predictor.epochs` to 30. Total time ~2–3 h on T4.